In [1]:
from augmentlib.pipelines.new_offline import NewOfflineAugmentedDatasetGenerator

In [2]:
from local_datasets.oxford_flower102 import OxfordFlowers102

In [3]:
flowers = OxfordFlowers102(root='./data', transform=None, download=True)

100%|██████████| 345M/345M [00:07<00:00, 48.4MB/s] 
100%|██████████| 502/502 [00:00<00:00, 2.15MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 31.1MB/s]


In [4]:
print(f"Train: {len(flowers.train_dataset)}")
print(f"Val:   {len(flowers.val_dataset)}")
print(f"Test:  {len(flowers.test_dataset)}")

Train: 1020
Val:   1020
Test:  6149


In [5]:
from augmentlib.methods.genmix import GenMixAugmentor

/workspace/Course_work_HSE_2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
genmix = GenMixAugmentor()

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 14.00it/s]


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:03<00:00, 99.0MB/s] 


[GenMix] Downloading fractal dataset from Kaggle: tomandjerry2005/fractal-mixing-set-pixmix


100%|██████████| 1.63G/1.63G [01:00<00:00, 28.9MB/s]

Extracting files...


[GenMix] Dataset downloaded to: /root/.cache/kagglehub/datasets/tomandjerry2005/fractal-mixing-set-pixmix/versions/1
[GenMix] Found 18937 fractal images
[GenMix] Loaded 18937 fractal paths (lazy loading)
[GenMix] Safety checker disabled. Max regeneration attempts: 5


In [7]:
generator = NewOfflineAugmentedDatasetGenerator(
    method=genmix,   # ваш объект AGA / GenMix / DA-Fusion
    save_dir="./augmented_train",
    num_aug=1,      # M = 10
    max_tries=20
)

In [8]:
generator.generate(flowers.train_dataset)

[GenMix] μ=0.0748, σ=0.1717


Generating augmented data: 100%|██████████| 1020/1020 [10:13<00:00,  1.66it/s]

Saved index to ./augmented_train/dataset_index.json


In [13]:
from training.dataset import MixedAugDataset

In [87]:
from utils.transforms import train_transform, val_test_transform

In [100]:
from torchvision import transforms

train_transforms = transforms.Compose([
    # transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    # transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [101]:
# import random
# from PIL import Image
from torch.utils.data import Dataset, DataLoader

In [102]:
train_set = MixedAugDataset(
    root="./augmented_train",
    index_path="./augmented_train/dataset_index.json",
    transform=train_transforms,
    alpha=0.5
)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

In [103]:
len(train_set)

1020

In [121]:
print(train_set[0])

(tensor([[[-1.5014, -1.5014, -1.4843,  ...,  0.0227,  0.0912,  0.0569],
         [-1.5185, -1.5014, -1.4843,  ...,  0.0056,  0.0227, -0.0116],
         [-1.5185, -1.5185, -1.5014,  ...,  0.0398,  0.0056, -0.0116],
         ...,
         [-1.7240, -1.6898, -1.6555,  ..., -0.4054, -0.4911, -0.4911],
         [-1.7240, -1.6898, -1.6555,  ..., -0.4397, -0.4911, -0.4911],
         [-1.7240, -1.6898, -1.6727,  ..., -0.4568, -0.4911, -0.4911]],

        [[-1.4580, -1.4580, -1.4405,  ..., -0.6877, -0.5826, -0.5651],
         [-1.4755, -1.4580, -1.4405,  ..., -0.7052, -0.6527, -0.6352],
         [-1.4755, -1.4755, -1.4580,  ..., -0.7227, -0.7052, -0.6877],
         ...,
         [-1.2304, -1.1954, -1.1604,  ..., -0.6527, -0.7577, -0.7577],
         [-1.2304, -1.1954, -1.1604,  ..., -0.6877, -0.7577, -0.7577],
         [-1.2304, -1.1954, -1.1779,  ..., -0.7052, -0.7577, -0.7577]],

        [[-1.3164, -1.3164, -1.2990,  ..., -0.2184, -0.1661, -0.1835],
         [-1.3339, -1.3164, -1.2990,  ..., -